# 延后初始化
:label:`sec_deferred_init`

到目前为止，我们忽略了建立网络时需要做的以下这些事情：

* 我们定义了网络架构，但没有指定输入维度。
* 我们添加层时没有指定前一层的输出维度。
* 我们在初始化参数时，甚至没有足够的信息来确定模型应该包含多少参数。

有些读者可能会对我们的代码能运行感到惊讶。
毕竟，深度学习框架无法判断网络的输入维度是什么。
这里的诀窍是框架的*延后初始化*（defers initialization），
即直到数据第一次通过模型传递时，框架才会动态地推断出每个层的大小。

在以后，当使用卷积神经网络时，
由于输入维度（即图像的分辨率）将影响每个后续层的维数，
有了该技术将更加方便。
现在我们在编写代码时无须知道维度是什么就可以设置参数，
这种能力可以大大简化定义和修改模型的任务。
接下来，我们将更深入地研究初始化机制。

## 实例化网络

首先，让我们实例化一个多层感知机。


In [11]:
import torch
from torch import nn  #从 PyTorch 中导入神经网络模块 torch.nn，并简写为 nn

torch.manual_seed(0)  #将 PyTorch 随机数生成器的种子设置为 0，使随机结果尽量可以复现

In [12]:
#seed()
#例如：torch.manual_seed(0)
    ## print(torch.rand(3))
    #每次重新启动内核并执行这两行，通常都会得到相同的随机数。
#LazyLinear 第一次前向传播时进行的权重初始化。
#因此设置随机种子后，按照相同顺序重新运行 Cell，输入数据和初始化参数通常能够保持一致。
#数字 0 没有特殊意义，也可以换成其他整数：
#torch.manual_seed(42)
#torch.manual_seed(2026)
#相同种子通常产生相同的随机数序列，不同种子产生不同序列。
#最后显示类似：<torch._C.Generator ...>
#这是因为 torch.manual_seed(0) 会返回随机数生成器对象，
#而 Jupyter 会自动显示代码 Cell 最后一个表达式的返回结果。它不是报错，也可以用分号隐藏：
#torch.manual_seed(0);

In [13]:
# LazyLinear只需要指定输出特征数，输入特征数将在第一次前向传播时推断
net = nn.Sequential(
    nn.LazyLinear(256),
    nn.ReLU(),
    nn.LazyLinear(10)
)

net

Sequential(
  (0): LazyLinear(in_features=0, out_features=256, bias=True)
  (1): ReLU()
  (2): LazyLinear(in_features=0, out_features=10, bias=True)
)

In [16]:
#普通全连接层通常写成：
# nn.Linear(in_features, out_features)
#但 LazyLinear 暂时不要求指定输入特征数，只需要指定输出特征数：
#nn.LazyLinear(256)
#输入特征数会在第一次调用：net(X)时自动匹配输入维度

此时，因为输入维数是未知的，所以网络不可能知道输入层权重的维数。
因此，框架尚未初始化任何参数，我们通过尝试访问以下参数进行确认。


In [18]:
# 前向传播前，权重和偏置都还是未初始化参数
for name, param in net.named_parameters():
    print(f'{name:8s}: {type(param).__name__}')

0.weight: UninitializedParameter
0.bias  : UninitializedParameter
2.weight: UninitializedParameter
2.bias  : UninitializedParameter


In [ ]:
print('第一层是否含有未初始化参数：',
      net[0].has_uninitialized_params())

# 未初始化参数还没有确定形状，直接访问shape会报错，所以用try捕获
try:
    print(net[0].weight.shape)
except RuntimeError as error:#意思是：如果 try 中发生了 RuntimeError，
            #不要让整个 Cell 停止，而是把错误对象保存到变量 error，然后执行下面的代码。
    print(type(error).__name__ + ':', error)
    #type取得错误对象的类型##.__name__取得类型名称字符串：#  #拼接成："RuntimeError:"
#
#
#使用 try/except 的目的不是修复错误，而是有意演示：前向传播之前，LazyLinear的参数还没有确定形状

第一层是否含有未初始化参数： True
RuntimeError: Can't access the shape of an uninitialized parameter or buffer. This error usually happens in `load_state_dict` when trying to load an uninitialized parameter into an initialized one. Call `forward` to initialize the parameters before accessing their attributes.


In [31]:
#同时避免这个预期中的错误中断 notebook。
#需要注意：这个 Cell 应该在第一次执行 net(X) 之前运行。
#执行net()过后，LazyLinear 会转变为普通 Linear，这时重新运行：
try:
    net[0].has_uninitialized_params()
except AttributeError as error:
    print(type(error).__name__ + ':', error)

#可能会得到 AttributeError，因为普通 Linear 没有这个方法。
#要重新观察延后初始化状态，需要重新运行创建 net 的第二个代码 Cell。

AttributeError: 'Linear' object has no attribute 'has_uninitialized_params'


接下来让我们将数据通过网络，最终使框架初始化参数。


In [25]:
# 每个样本有20个输入特征；第一次调用会触发延后初始化
X = torch.rand(2, 20)
Y = net(X)

Y.shape

torch.Size([2, 10])

In [32]:
# LazyLinear已经根据输入自动转换成了具有确定尺寸的Linear
print(net)
print()
for name, param in net.named_parameters():
    print(f'{name:8s}: {tuple(param.shape)}')

Sequential(
  (0): Linear(in_features=20, out_features=256, bias=True)
  (1): ReLU()
  (2): Linear(in_features=256, out_features=10, bias=True)
)

0.weight: (256, 20)
0.bias  : (256,)
2.weight: (10, 256)
2.bias  : (10,)


In [47]:
# 推荐在试运行完成、参数形状确定后再创建优化器
optimizer = torch.optim.SGD(net.parameters(), lr=0.1)
len(optimizer.param_groups[0]['params'])#这里的 4 表示四个参数张量，不是四个数值。

4

In [48]:
#创建一个随机梯度下降优化器：
#torch.optim.SGD
#SGD 会在训练过程中根据参数梯度更新模型参数。
#它本身不会立即训练模型，只是先记录需要管理的参数以及学习率等配置。

#net.parameters()
#递归取得网络中所有可训练参数。
#当前网络经过试运行后相当于：
#nn.Sequential(
#    nn.Linear(20, 256),
#    nn.ReLU(),
#    nn.Linear(256, 10)
#)

In [53]:
#为什么要在试运行后创建sgd优化器？
#前向传播前，LazyLinear 的参数是：UninitializedParameter
#此时参数的形状还未知，一些优化器操作、参数统计或状态初始化可能无法正常处理它们。

#net()后参数被正式创建
#这时再创建优化器，优化器拿到的是形状已经确定的参数，流程更加可靠：
#创建Lazy模型
# 移动到指定设备和数据类型
# 输入示例数据进行试运行
# 参数形状确定
# 创建优化器
# 正式训练

#查看优化器管理了多少个参数张量
print(len(optimizer.param_groups[0]['params']), '这里的 4 表示四个参数张量，不是四个数值。')
#优化器把参数按“参数组”管理。它是一个列表：optimizer.param_groups
#不同参数组可以设置不同的学习率，例如：
#[
#    {"params": 第一组参数, "lr": 0.1},
#    {"params": 第二组参数, "lr": 0.01}
#]
#当前只传入了：net.parameters()
#所以只有一个参数组：optimizer.param_groups[0]
#取得第一个参数组管理的参数张量列表：
print('第一个参数组管理的参数张量列表：', optimizer.param_groups[0]['params'])

4 这里的 4 表示四个参数张量，不是四个数值。
第一个参数组管理的参数张量列表： [Parameter containing:
tensor([[ 0.0118, -0.1146,  0.0378,  ..., -0.1734, -0.1550, -0.1155],
        [ 0.1012,  0.0899, -0.1325,  ..., -0.1944,  0.0206, -0.1399],
        [-0.2084,  0.1987,  0.1700,  ...,  0.0468, -0.1744, -0.1288],
        ...,
        [-0.0346, -0.1717,  0.0738,  ...,  0.0745,  0.1138,  0.0234],
        [-0.1108, -0.2147,  0.1209,  ...,  0.0682,  0.1993,  0.0780],
        [-0.0792,  0.0975,  0.1905,  ..., -0.0579, -0.0916,  0.0886]],
       requires_grad=True), Parameter containing:
tensor([ 0.0775, -0.0072,  0.0356,  0.0768, -0.2161, -0.1977, -0.0595,  0.1523,
         0.1091, -0.2134, -0.0671, -0.0236,  0.0734,  0.1829,  0.0049, -0.1741,
         0.0137, -0.0863, -0.1367,  0.1239, -0.0300,  0.1057,  0.1741,  0.0220,
         0.1259, -0.1056,  0.1248,  0.0089,  0.1486,  0.2138, -0.1691,  0.2123,
         0.1707, -0.1801,  0.0655, -0.1820,  0.0725,  0.0346, -0.2135,  0.0812,
        -0.0614,  0.2229,  0.2182,  0.0772,  0.1763

In [46]:
#模型的参数数值总量可以这样计算：
sum(param.numel() for param in net.parameters())
#当前模型共有：256*20 + 256 + 10*256 + 10 = 7946个参数数值

7946

In [45]:
#后续经典训练步骤
#optimizer.zero_grad()  # 清除上一轮梯度

#Y_hat = net(X)         # 前向传播
#loss = loss_fn(Y_hat, y)

#loss.backward()        # 反向传播，计算梯度
#optimizer.step()       # 根据梯度更新参数

一旦我们知道输入维数是20，框架可以通过代入值20来识别第一层权重矩阵的形状。
识别出第一层的形状后，框架处理第二层，依此类推，直到所有形状都已知为止。
注意，在这种情况下，只有第一层需要延迟初始化，但是框架仍是按顺序初始化的。
等到知道了所有的参数形状，框架就可以初始化参数。

## 小结

* 延后初始化使框架能够自动推断参数形状，使修改模型架构变得容易，避免了一些常见的错误。
* 我们可以通过模型传递数据，使框架最终初始化参数。

## 练习

1. 如果指定了第一层的输入尺寸，但没有指定后续层的尺寸，会发生什么？是否立即进行初始化？
1. 如果指定了不匹配的维度会发生什么？
1. 如果输入具有不同的维度，需要做什么？提示：查看参数绑定的相关内容。


[Discussions](https://discuss.d2l.ai/t/5770)
